# DTIAM benchmark results — sanity check

Validates the real k-fold benchmark CSVs (produced by `train_dtiam_models.py` **without** `--ensemble-only`,
i.e. actual held-out evaluation per fold — not the `--ensemble-only` production-model training path, which
never evaluates on held-out data).

- **DTA / Davis**: 5-fold, `warm_start` split — RMSE, MSE, Pearson, Spearman, CI (pKd scale).
- **DTI / Yamanishi_08**: 10-fold, `warm_start` split — AUROC, AUPR.

Each section loads its CSV, checks it's well-formed (right fold count, no missing values, metrics within
valid ranges, RMSE/MSE internally consistent), and reports per-fold + mean values.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Notebook lives in code/, benchmark results live in ../results/benchmark/
RESULTS_DIR = Path("..") / "results" / "benchmark"

pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [ ]:
def check_regression_results(df: pd.DataFrame, expected_folds: int, name: str) -> None:
    """Sanity-check a DTA-style k-fold results CSV (RMSE, MSE, Pearson, Spearman, CI)."""
    problems = []

    if len(df) != expected_folds:
        problems.append(f"expected {expected_folds} folds, got {len(df)}")
    if df.isna().any().any():
        problems.append(f"contains NaNs:\n{df[df.isna().any(axis=1)]}")
    if sorted(df["fold"].tolist()) != list(range(1, expected_folds + 1)):
        problems.append(f"fold column isn't a clean 1..{expected_folds} sequence: {sorted(df['fold'].tolist())}")

    for col in ["RMSE", "MSE", "CI"]:
        if (df[col] < 0).any():
            problems.append(f"{col} has negative values (should be >= 0)")
    for col in ["Pearson", "Spearman"]:
        if not df[col].between(-1, 1).all():
            problems.append(f"{col} has values outside [-1, 1]")
    if not df["CI"].between(0, 1).all():
        problems.append("CI has values outside [0, 1]")

    # RMSE should equal sqrt(MSE) -- if not, the two columns were computed inconsistently.
    implied_rmse = np.sqrt(df["MSE"])
    if not np.allclose(df["RMSE"], implied_rmse, atol=1e-6):
        max_diff = (df["RMSE"] - implied_rmse).abs().max()
        problems.append(f"RMSE != sqrt(MSE) for at least one fold (max diff {max_diff:.2e})")

    _report(name, df, problems)


def check_classification_results(df: pd.DataFrame, expected_folds: int, name: str) -> None:
    """Sanity-check a DTI/MOA-style k-fold results CSV (AUROC, AUPR)."""
    problems = []

    if len(df) != expected_folds:
        problems.append(f"expected {expected_folds} folds, got {len(df)}")
    if df.isna().any().any():
        problems.append(f"contains NaNs:\n{df[df.isna().any(axis=1)]}")
    if sorted(df["fold"].tolist()) != list(range(1, expected_folds + 1)):
        problems.append(f"fold column isn't a clean 1..{expected_folds} sequence: {sorted(df['fold'].tolist())}")

    for col in ["AUROC", "AUPR"]:
        if not df[col].between(0, 1).all():
            problems.append(f"{col} has values outside [0, 1]")
        if (df[col] == 1.0).any():
            problems.append(f"{col} is exactly 1.0 for at least one fold -- worth double-checking for leakage")

    _report(name, df, problems)


def _report(name: str, df: pd.DataFrame, problems: list) -> None:
    print(f"=== {name} ({len(df)} folds) ===")
    display(df)
    means = df.drop(columns="fold").mean()
    stds = df.drop(columns="fold").std()
    print("\nMean ± std across folds:")
    for col in means.index:
        rel_std = stds[col] / means[col] * 100 if means[col] != 0 else float("nan")
        print(f"  {col:10s} {means[col]:.4f} ± {stds[col]:.4f}  (rel. std {rel_std:.1f}%)")

    if problems:
        print(f"\n⚠ {len(problems)} issue(s) found:")
        for p in problems:
            print(f"  - {p}")
    else:
        print("\n✓ All sanity checks passed.")
    print()

## DTA — Davis (5-fold, warm_start)

In [ ]:
davis_df = pd.read_csv(RESULTS_DIR / "dta_davis" / "dta_davis_results.csv")
check_regression_results(davis_df, expected_folds=5, name="DTA / Davis")

## DTI — Yamanishi_08 (10-fold, warm_start)

In [ ]:
yamanishi_df = pd.read_csv(RESULTS_DIR / "dti_yamanishi_08" / "dti_yamanishi_08_results.csv")
check_classification_results(yamanishi_df, expected_folds=10, name="DTI / Yamanishi_08")

## Per-fold stability (visual check)

Low fold-to-fold variance is a good sign the model generalizes consistently within its own domain,
rather than getting lucky on one particular split.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(davis_df["fold"], davis_df["Pearson"], "o-", label="Pearson")
axes[0].plot(davis_df["fold"], davis_df["CI"], "s-", label="CI")
axes[0].plot(davis_df["fold"], davis_df["Spearman"], "^-", label="Spearman")
axes[0].set_title("DTA / Davis")
axes[0].set_xlabel("fold")
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(yamanishi_df["fold"], yamanishi_df["AUROC"], "o-", label="AUROC")
axes[1].plot(yamanishi_df["fold"], yamanishi_df["AUPR"], "s-", label="AUPR")
axes[1].set_title("DTI / Yamanishi_08")
axes[1].set_xlabel("fold")
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Diversidad estructural de los candidatos generados (Tanimoto + Shannon entropy)

Sobre `results/generation/acaricide/top_candidates.csv` (candidatos del GA, top-20 por target).

- **Diversidad interna (Tanimoto)**: `1 - similitud de Tanimoto promedio por pares` usando fingerprints
  ECFP4 (Morgan, radio 2, 2048 bits). Más alto = candidatos más diversos entre sí dentro de ese target.
  Métrica estándar en literatura de generación de novo (IntDiv, MOSES/GuacaMol benchmark).
- **Shannon entropy normalizada (scaffolds)**: entropía de la distribución de scaffolds de
  Bemis-Murcko entre los candidatos de un target, normalizada por `log(N)` para caer en [0, 1] —
  1.0 = cada candidato tiene un scaffold único, 0.0 = todos comparten el mismo scaffold.

In [ ]:
from collections import Counter

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold

RDLogger.DisableLog("rdApp.*")

candidates_df = pd.read_csv(Path("..") / "results" / "generation" / "acaricide" / "top_candidates.csv")
print(f"Loaded {len(candidates_df)} candidates across {candidates_df['target'].nunique()} targets")
candidates_df.head()

In [ ]:
def get_fingerprint(smiles: str, radius: int = 2, n_bits: int = 2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)


def internal_diversity(smiles_list: list) -> float:
    """1 - mean pairwise Tanimoto similarity (ECFP4) among a set of molecules.
    Higher = more structurally diverse (standard MOSES/GuacaMol 'IntDiv' metric)."""
    fps = [fp for fp in (get_fingerprint(s) for s in smiles_list) if fp is not None]
    if len(fps) < 2:
        return float("nan")
    sims = []
    for i in range(len(fps) - 1):
        sims.extend(DataStructs.BulkTanimotoSimilarity(fps[i], fps[i + 1:]))
    return 1 - float(np.mean(sims))


def scaffold_entropy_normalized(smiles_list: list) -> float:
    """Shannon entropy of the Bemis-Murcko scaffold distribution, normalized by
    log(N) so a fully unique-scaffold set scores 1.0 and a single shared
    scaffold scores 0.0. Acyclic molecules collapse to the empty-string scaffold
    (RDKit's Murcko scaffold is undefined without a ring), which is itself a
    valid, informative bucket -- it means "no ring system" as a category."""
    scaffolds = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        scaffolds.append(Chem.MolToSmiles(scaffold))

    n = len(scaffolds)
    if n < 2:
        return float("nan")
    counts = np.array(list(Counter(scaffolds).values()))
    probs = counts / n
    entropy = -np.sum(probs * np.log(probs))
    max_entropy = np.log(n)
    return float(entropy / max_entropy)

In [ ]:
rows = []
for target, group in candidates_df.groupby("target"):
    smiles_list = group["smiles"].tolist()
    rows.append({
        "target": target,
        "n_candidates": len(smiles_list),
        "internal_diversity": internal_diversity(smiles_list),
        "scaffold_entropy_norm": scaffold_entropy_normalized(smiles_list),
    })

diversity_df = pd.DataFrame(rows).sort_values("target").reset_index(drop=True)
display(diversity_df)

print("\nResumen across all 53 targets (one row per target):")
display(diversity_df[["internal_diversity", "scaffold_entropy_norm"]].describe())

In [ ]:
# Global (todos los candidatos juntos, sin separar por target) -- útil como número
# único para la slide, aunque mezcla química de las 53 dianas.
all_smiles = candidates_df["smiles"].tolist()
print("=== Global (todos los candidatos agrupados) ===")
print(f"Diversidad interna (Tanimoto): {internal_diversity(all_smiles):.4f}")
print(f"Shannon entropy normalizada (scaffolds): {scaffold_entropy_normalized(all_smiles):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(diversity_df["internal_diversity"].dropna(), bins=15, color="steelblue", edgecolor="black")
axes[0].set_title("Diversidad interna (Tanimoto) por target")
axes[0].set_xlabel("1 - Tanimoto promedio")
axes[0].set_ylabel("Número de targets")

axes[1].hist(diversity_df["scaffold_entropy_norm"].dropna(), bins=15, color="darkorange", edgecolor="black")
axes[1].set_title("Shannon entropy normalizada (scaffolds) por target")
axes[1].set_xlabel("Entropía normalizada")
axes[1].set_ylabel("Número de targets")

plt.tight_layout()
plt.show()